In [6]:
import colorsys
import numpy as np
import pandas as pd
import dataframe_image as dfi

Load the WIS and rank the models:

In [2]:
m_wis_norm = pd.read_csv('results/metrics_wis_norm.csv.gz')

m_wis_norm = m_wis_norm.groupby(['model', 'state'], as_index=False)['WIS'].mean()

m_wis_norm['rank'] = m_wis_norm.groupby('state')['WIS'].rank(method='first').astype(int)

m_top3 = m_wis_norm[m_wis_norm['rank'] <= 3]

ranked_models = m_top3.pivot(index='rank', columns='state', values='model')

ranked_models

state,AC,AL,AM,AP,BA,CE,DF,ES,GO,MA,...,PR,RJ,RN,RO,RR,RS,SC,SE,SP,TO
rank,,,,,,,,,,,,,,,,,,,,,
1,144,135,144,158,136,133,157,154,135,135,...,136,156,135,150,133,156,155,145,155,136
2,133,155,150,143,135,152,156,158,136,158,...,144,133,152,157,144,155,145,150,135,150
3,158,150,136,157,156,157,136,157,144,144,...,155,143,158,158,136,157,135,158,156,156


Definindo uma paleta de cores para os modelos que aparecem no top 3: 

In [3]:
dict_models_name = {157: 'UERJ-SARIMAX-2', 156: 'Dengue oracle M2', 155: 'Dengue Oracle M1', 135:'GHR Model', 
                    133:'Chronos-Bolt', 158: 'Cornell PEH', 143: 'Model fourier-\ngravidade', 144:'Beat it', 
                    150: 'LNCC-AR_p-1', 
        145: 'CNNLSTM' , 141: 'Kaust Geohealth', 136: 'Imperial-TFT Model', 137: 'LSTM-RF model', 
                    138: 'TSMixer ZKI-PH4', 154: 'LNCC-SURGE-1', 152: 'LNCC-CLIDENGO-1', 134: 'ISI_Dengue_Model', 
                    108:'IMPA-TECH'} 

keys = np.unique(ranked_models.values)
n = len(keys)

# Gerar cores bem espaçadas no círculo HSV
def generate_vibrant_colors(n, s=0.8, v=0.9):
    hues = np.linspace(0, 1, n, endpoint=False)
    colors = [colorsys.hsv_to_rgb(h, s, v) for h in hues]
    return colors

# Função para converter RGB [0,1] para HEX
def rgb_to_hex(rgb):
    return '#{:02x}{:02x}{:02x}'.format(
        int(rgb[0]*255), int(rgb[1]*255), int(rgb[2]*255)
    )

colors = generate_vibrant_colors(n)

# Criar dict {key: HEX color}
color_palette = {dict_models_name[k]: rgb_to_hex(colors[i]) for i, k in enumerate(keys)}

color_palette

{'Chronos-Bolt': '#e52d2d',
 'GHR Model': '#e5822d',
 'Imperial-TFT Model': '#e5d72d',
 'Model fourier-\ngravidade': '#9ee52d',
 'Beat it': '#4ae52d',
 'CNNLSTM': '#2de566',
 'LNCC-AR_p-1': '#2de5bb',
 'LNCC-CLIDENGO-1': '#2dbbe5',
 'LNCC-SURGE-1': '#2d66e5',
 'Dengue Oracle M1': '#4a2de5',
 'Dengue oracle M2': '#9e2de5',
 'UERJ-SARIMAX-2': '#e52dd7',
 'Cornell PEH': '#e52d82'}

Funções para aplicar os estilos e e criar o quadro: 

In [ ]:
def style_cells(val, 
                cell_colors = color_palette):
    """background das células de dados (mantém texto em negrito)."""
    bg = cell_colors.get(val, "white")
    return f"background-color: {bg}; font-weight: bold; text-align: center; color: black;"

def style_medal(val, medal_colors = {
    "GOLD":   "#DAA520",   # dourado
    "SILVER": "#B0B0B0",   # prata
    "BRONZE": "#A97142"    # bronze
}):
    """aplica cor apenas ao texto da coluna MEDAL."""
    c = medal_colors.get(val, "black")
    return f"color: {c}; font-weight: bold; text-align: left;"

def gen_table(ranked_models, dict_models_name = dict_models_name, index = ["GOLD", "SILVER", "BRONZE"]): 

    data = ranked_models
    data = data.replace(dict_models_name)

    df = pd.DataFrame(data.values, index=index, columns = data.columns)

    df.index.name = "MEDAL"

    df_reset = df.reset_index()   # agora há uma coluna "MEDAL" + as colunas originais

    styler = df_reset.style

    # estiliza todas as células de dados (todas colunas exceto 'MEDAL'):
    data_cols = [c for c in df_reset.columns if c != "MEDAL"]
    styler = styler.applymap(style_cells, subset=pd.IndexSlice[:, data_cols])

    # estiliza apenas a coluna MEDAL (texto colorido)
    styler = styler.applymap(style_medal, subset=pd.IndexSlice[:, ["MEDAL"]])

    # estilos gerais da tabela (bordas, fonte, cabeçalho)
    styler = styler.set_table_styles([
        {"selector": "th", "props": [("font-weight", "bold"), ("text-align", "center"), ("border", "1px solid black")]},
        {"selector": "td", "props": [("border", "1px solid black"), ("padding", "6px")]},
        {"selector": "table", "props": [("border-collapse", "collapse"), ("font-family", "Arial"), ("font-size", "13px")]}
    ])

    styler = styler.hide(axis="index")  # hides the index completely

    return styler


In [9]:
order = {
    'south': ['PR', 'RS', 'SC'],
    'southeast': ['ES', 'MG', 'RJ', 'SP'],
    'midwest': ['DF', 'GO', 'MT', 'MS'],
    'northeast': ['AL', 'BA', 'CE', 'MA', 'PB', 'PE', 'PI', 'RN', 'SE'],
    'north': ['AC', 'AM', 'AP', 'PA', 'RO', 'RR', 'TO'] } 

In [12]:
for region in order.keys(): 
    styler = gen_table(ranked_models[order[region]],) 
    dfi.export(styler, f"figures/medals_{region}.png", dpi = 400)
    

/var/folders/ch/kxpr39wx44v97968yr_4hmch0000gn/T/ipykernel_29400/732961857.py:31: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  styler = styler.applymap(style_cells, subset=pd.IndexSlice[:, data_cols])
/var/folders/ch/kxpr39wx44v97968yr_4hmch0000gn/T/ipykernel_29400/732961857.py:34: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  styler = styler.applymap(style_medal, subset=pd.IndexSlice[:, ["MEDAL"]])
/var/folders/ch/kxpr39wx44v97968yr_4hmch0000gn/T/ipykernel_29400/732961857.py:31: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  styler = styler.applymap(style_cells, subset=pd.IndexSlice[:, data_cols])
/var/folders/ch/kxpr39wx44v97968yr_4hmch0000gn/T/ipykernel_29400/732961857.py:34: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  styler = styler.applymap(style_medal, subset=pd.IndexSlice[:, ["MEDAL"]])
/var/folders/ch/kxpr39wx44v97968yr_4hmch0000gn/T/ipykernel_29400

In [13]:
ranked_models

state,AC,AL,AM,AP,BA,CE,DF,ES,GO,MA,...,PR,RJ,RN,RO,RR,RS,SC,SE,SP,TO
rank,,,,,,,,,,,,,,,,,,,,,
1,144,135,144,158,136,133,157,154,135,135,...,136,156,135,150,133,156,155,145,155,136
2,133,155,150,143,135,152,156,158,136,158,...,144,133,152,157,144,155,145,150,135,150
3,158,150,136,157,156,157,136,157,144,144,...,155,143,158,158,136,157,135,158,156,156


In [45]:
agg_medals = m_top3.groupby(['model', 'rank'])[['WIS']].count().reset_index().pivot(index = 'model',
                                                                       columns = 'rank',
                                                                        values = 'WIS' ).fillna(0).sort_values(by =1, ascending = False)

agg_medals = agg_medals.astype(int).reset_index().rename(columns = {
        1: 'GOLD',
        2: 'SILVER', 
        3: 'BRONZE',
        'model': 'Model'
})

agg_medals['Model'] = agg_medals['Model'].replace(dict_models_name)

agg_medals['TOTAL'] = agg_medals['GOLD'] + agg_medals['SILVER'] + agg_medals['BRONZE']

styler = agg_medals.style

# estiliza todas as células de dados (todas colunas exceto 'MEDAL'):
styler = styler.applymap(style_cells, subset=pd.IndexSlice[:, 'Model'])

# estiliza apenas a coluna MEDAL (texto colorido)
#styler = styler.applymap(style_medal, subset=pd.IndexSlice[:, ["MEDAL"]])

# estilos gerais da tabela (bordas, fonte, cabeçalho)
styler = styler.set_table_styles([
        {"selector": "th", "props": [("font-weight", "bold"), ("text-align", "center"), ("border", "1px solid black")]},
        {"selector": "td", "props": [("border", "1px solid black"), ("padding", "6px")]},
        {"selector": "table", "props": [("border-collapse", "collapse"), ("font-family", "Arial"), ("font-size", "13px")]}
    ])


styler = styler.set_table_styles([
    {'selector': 'th.col1', 'props': [('color',"#DAA520"), ('font-weight', 'bold')]},
    {'selector': 'th.col2', 'props': [('color', '#B0B0B0'), ('font-weight', 'bold')]},
    {'selector': 'th.col3', 'props': [('color', '#A97142'), ('font-weight', 'bold')]}, 
    {'selector': 'th.col_heading', 'props': [('text-align', 'center')]},
    {'selector': 'td', 'props': [('text-align', 'center')]} 
])

styler = styler.hide(axis="index")  # hides the index completely
styler

/var/folders/ch/kxpr39wx44v97968yr_4hmch0000gn/T/ipykernel_29400/906312614.py:19: FutureWarning: Styler.applymap has been deprecated. Use Styler.map instead.
  styler = styler.applymap(style_cells, subset=pd.IndexSlice[:, 'Model'])


Model,GOLD,SILVER,BRONZE,TOTAL
LNCC-AR_p-1,5,3,1,9
GHR Model,4,2,2,8
Dengue oracle M2,4,2,3,9
Imperial-TFT Model,3,1,3,7
Dengue Oracle M1,3,3,2,8
Chronos-Bolt,2,3,0,5
Beat it,2,2,2,6
CNNLSTM,1,1,1,3
LNCC-SURGE-1,1,1,0,2
UERJ-SARIMAX-2,1,1,7,9
